In [1]:
# Cell 1: NLP Interface PoC — Natural Language → malariagen_data API
# 
# Goal: Demonstrate how plain-language questions from researchers
# can be translated into executable malariagen_data API calls.
#
# Example:
#   User: "Show me allele frequencies for Vgsc in Kenya"
#   → ag3.plot_frequencies_heatmap(transcript="AGAP004707-RD", 
#                                   area=Region("KE"), ...)
#
# This PoC uses rule-based NLP (intent + entity extraction) to show
# the concept. A full implementation would use fine-tuned LLMs.

import re
import json
from dataclasses import dataclass, field
from typing import Optional, List, Dict, Any

print("NLP Interface PoC — Natural Language → malariagen_data API")
print("=" * 60)

NLP Interface PoC — Natural Language → malariagen_data API


In [2]:
# Cell 2: API Method Registry
# This mirrors what describe_api() returns + parameter-level detail
# that an NLP system needs to generate valid calls.

API_METHODS = {
    "plot_frequencies_heatmap": {
        "description": "Plot a heatmap of allele frequencies for SNPs in a gene transcript",
        "category": "plot",
        "parameters": {
            "transcript": {"type": "str", "description": "Gene transcript ID (e.g., AGAP004707-RD for Vgsc)"},
            "sample_sets": {"type": "str|list", "description": "Sample set(s) to include", "optional": True},
            "sample_query": {"type": "str", "description": "Pandas query to filter samples", "optional": True},
            "area": {"type": "Region", "description": "Geographic area to filter by"},
            "min_cohort_size": {"type": "int", "description": "Minimum samples per cohort", "default": 10},
        },
        "keywords": ["frequency", "frequencies", "allele", "heatmap", "snp", "mutation", "variant"],
    },
    "plot_frequencies_time_series": {
        "description": "Plot allele frequencies over time for SNPs in a gene",
        "category": "plot",
        "parameters": {
            "transcript": {"type": "str", "description": "Gene transcript ID"},
            "area": {"type": "Region", "description": "Geographic area"},
            "period_by": {"type": "str", "description": "Time period grouping (year, quarter, month)", "default": "year"},
        },
        "keywords": ["time", "trend", "temporal", "over time", "change", "year", "series"],
    },
    "pairwise_average_fst": {
        "description": "Compute pairwise average Fst between cohorts",
        "category": "analysis",
        "parameters": {
            "contig": {"type": "str", "description": "Chromosome/contig name (e.g., 2L, 3R, X)"},
            "cohorts": {"type": "str|dict", "description": "Cohort column or dict mapping names to queries"},
            "sample_sets": {"type": "str|list", "description": "Sample set(s)", "optional": True},
            "min_cohort_size": {"type": "int", "description": "Minimum samples per cohort", "default": 10},
        },
        "keywords": ["fst", "differentiation", "genetic distance", "divergence", "population structure"],
    },
    "plot_pairwise_average_fst": {
        "description": "Plot a heatmap of pairwise Fst values between cohorts",
        "category": "plot",
        "parameters": {
            "fst_df": {"type": "DataFrame", "description": "Output from pairwise_average_fst()"},
            "annotation": {"type": "str", "description": "Upper triangle content: None, 'standard error', 'Z score', 'lower triangle'", "optional": True},
        },
        "keywords": ["fst", "heatmap", "pairwise", "plot", "visualize", "compare populations"],
    },
    "plot_snps_dxy": {
        "description": "Plot absolute genetic divergence (Dxy) across a contig",
        "category": "plot",
        "parameters": {
            "contig": {"type": "str", "description": "Chromosome/contig"},
            "cohort1_query": {"type": "str", "description": "Query for first cohort"},
            "cohort2_query": {"type": "str", "description": "Query for second cohort"},
            "window_size": {"type": "int", "description": "Window size in base pairs", "default": 20000},
        },
        "keywords": ["dxy", "divergence", "absolute", "between populations"],
    },
    "sample_metadata": {
        "description": "Access sample metadata including collection location, date, species, and partner",
        "category": "data",
        "parameters": {
            "sample_sets": {"type": "str|list", "description": "Sample set(s) to include", "optional": True},
            "sample_query": {"type": "str", "description": "Pandas query to filter", "optional": True},
        },
        "keywords": ["metadata", "samples", "collection", "location", "country", "species", "info"],
    },
    "snp_calls": {
        "description": "Access SNP genotype calls for a genomic region",
        "category": "data",
        "parameters": {
            "region": {"type": "str", "description": "Genomic region (contig or contig:start-end)"},
            "sample_sets": {"type": "str|list", "description": "Sample set(s)", "optional": True},
            "sample_query": {"type": "str", "description": "Pandas query to filter", "optional": True},
        },
        "keywords": ["snp", "genotype", "calls", "variants", "data", "raw"],
    },
}

print(f"Registered {len(API_METHODS)} API methods:")
for name, info in API_METHODS.items():
    print(f"  [{info['category']:>8}] {name}: {info['description'][:60]}...")

Registered 7 API methods:
  [    plot] plot_frequencies_heatmap: Plot a heatmap of allele frequencies for SNPs in a gene tran...
  [    plot] plot_frequencies_time_series: Plot allele frequencies over time for SNPs in a gene...
  [analysis] pairwise_average_fst: Compute pairwise average Fst between cohorts...
  [    plot] plot_pairwise_average_fst: Plot a heatmap of pairwise Fst values between cohorts...
  [    plot] plot_snps_dxy: Plot absolute genetic divergence (Dxy) across a contig...
  [    data] sample_metadata: Access sample metadata including collection location, date, ...
  [    data] snp_calls: Access SNP genotype calls for a genomic region...


In [3]:
# Cell 3: Entity Dictionaries — Domain Knowledge for Parsing
# These map natural language terms to valid API parameter values.

# Gene name → transcript ID mapping
GENE_TRANSCRIPTS = {
    "vgsc": "AGAP004707-RD",
    "voltage-gated sodium channel": "AGAP004707-RD",
    "kdr": "AGAP004707-RD",
    "knockdown resistance": "AGAP004707-RD",
    "rdl": "AGAP006028-RA",
    "ace1": "AGAP001356-RA",
    "acetylcholinesterase": "AGAP001356-RA",
    "cyp6aa1": "AGAP002862-RA",
    "cyp6p3": "AGAP002865-RB",
    "cyp9k1": "AGAP000818-RA",
    "gste2": "AGAP009194-RA",
}

# Country → ISO code / area mapping
COUNTRY_AREAS = {
    "kenya": "KE",
    "tanzania": "TZ",
    "uganda": "UG",
    "ghana": "GH",
    "burkina faso": "BF",
    "cameroon": "CM",
    "mali": "ML",
    "mozambique": "MZ",
    "democratic republic of congo": "CD",
    "drc": "CD",
    "ethiopia": "ET",
    "nigeria": "NG",
    "senegal": "SN",
    "guinea": "GN",
    "ivory coast": "CI",
    "cote d'ivoire": "CI",
    "angola": "AO",
    "malawi": "MW",
    "zambia": "ZM",
    "benin": "BJ",
    "gabon": "GA",
    "gambia": "GM",
    "east africa": ["KE", "TZ", "UG", "ET", "MZ", "MW"],
    "west africa": ["GH", "BF", "ML", "NG", "SN", "GN", "CI", "BJ", "GM", "GA"],
}

# Chromosome/contig aliases
CONTIG_ALIASES = {
    "chromosome 2l": "2L",
    "chromosome 2r": "2R",
    "chromosome 3l": "3L",
    "chromosome 3r": "3R",
    "chromosome x": "X",
    "chr 2l": "2L", "chr 2r": "2R", "chr 3l": "3L", "chr 3r": "3R", "chr x": "X",
    "2l": "2L", "2r": "2R", "3l": "3L", "3r": "3R", "x": "X",
}

# Species name normalization
SPECIES_MAP = {
    "gambiae": "gambiae",
    "coluzzii": "coluzzii",
    "arabiensis": "arabiensis",
    "an. gambiae": "gambiae",
    "an. coluzzii": "coluzzii",
    "an. arabiensis": "arabiensis",
    "anopheles gambiae": "gambiae",
    "anopheles coluzzii": "coluzzii",
}

print(f"Knowledge base loaded:")
print(f"  {len(GENE_TRANSCRIPTS)} gene/transcript mappings")
print(f"  {len(COUNTRY_AREAS)} country/region mappings")
print(f"  {len(CONTIG_ALIASES)} contig aliases")
print(f"  {len(SPECIES_MAP)} species mappings")

Knowledge base loaded:
  11 gene/transcript mappings
  24 country/region mappings
  15 contig aliases
  8 species mappings


In [4]:
# Cell 4: NLP Engine — Intent Classification + Entity Extraction

@dataclass
class ParsedQuery:
    """Structured representation of a parsed natural language query."""
    raw_query: str
    intent: str  # The API method to call
    confidence: float  # 0-1 confidence score
    entities: Dict[str, Any] = field(default_factory=dict)
    api_call: str = ""  # Generated Python code
    explanation: str = ""  # Human-readable explanation

def classify_intent(query: str) -> tuple:
    """Match a query to the most relevant API method using keyword scoring."""
    query_lower = query.lower()
    scores = {}
    
    for method_name, method_info in API_METHODS.items():
        score = 0
        keywords = method_info["keywords"]
        
        for keyword in keywords:
            if keyword in query_lower:
                # Longer keyword matches get higher weight
                score += len(keyword.split())
        
        # Boost for exact method category hints
        if "plot" in query_lower or "show" in query_lower or "visualize" in query_lower:
            if method_info["category"] == "plot":
                score += 1
        if "compute" in query_lower or "calculate" in query_lower:
            if method_info["category"] == "analysis":
                score += 1
        if "get" in query_lower or "access" in query_lower or "list" in query_lower:
            if method_info["category"] == "data":
                score += 1
                
        # Context-specific boosts
        if "over time" in query_lower or "trend" in query_lower:
            if method_name == "plot_frequencies_time_series":
                score += 3
        if "compare" in query_lower and "population" in query_lower:
            if "fst" in method_name:
                score += 2
        if "heatmap" in query_lower and "fst" in query_lower:
            if method_name == "plot_pairwise_average_fst":
                score += 3
                
        scores[method_name] = score
    
    if not scores or max(scores.values()) == 0:
        return "unknown", 0.0
    
    best_method = max(scores, key=scores.get)
    max_score = scores[best_method]
    # Normalize confidence (cap at 1.0)
    confidence = min(max_score / 5.0, 1.0)
    
    return best_method, confidence

def extract_entities(query: str) -> Dict[str, Any]:
    """Extract structured entities from a natural language query."""
    query_lower = query.lower()
    entities = {}
    
    # Extract gene/transcript
    for gene_name, transcript_id in GENE_TRANSCRIPTS.items():
        if gene_name in query_lower:
            entities["gene"] = gene_name
            entities["transcript"] = transcript_id
            break
    
    # Extract country/area
    for country_name, area_code in COUNTRY_AREAS.items():
        if country_name in query_lower:
            entities["country"] = country_name
            entities["area"] = area_code
            break
    
    # Extract contig/chromosome
    for alias, contig in CONTIG_ALIASES.items():
        if alias in query_lower:
            entities["contig"] = contig
            break
    
    # Extract species
    for species_name, normalized in SPECIES_MAP.items():
        if species_name in query_lower:
            entities["species"] = normalized
            break
    
    # Extract cohort-related hints
    if "by country" in query_lower:
        entities["cohort_by"] = "country"
    elif "by species" in query_lower or "by taxon" in query_lower:
        entities["cohort_by"] = "taxon"
    elif "by region" in query_lower:
        entities["cohort_by"] = "admin1_iso"
    
    # Extract annotation preferences
    if "lower triangle" in query_lower:
        entities["annotation"] = "lower triangle"
    elif "standard error" in query_lower or "with errors" in query_lower:
        entities["annotation"] = "standard error"
    elif "z score" in query_lower or "z-score" in query_lower:
        entities["annotation"] = "Z score"
    
    return entities

def generate_api_call(method: str, entities: Dict[str, Any]) -> tuple:
    """Generate executable Python code from method + entities."""
    
    if method == "plot_frequencies_heatmap":
        transcript = entities.get("transcript", "AGAP004707-RD")
        gene = entities.get("gene", "Vgsc")
        area = entities.get("area", None)
        
        params = [f'transcript="{transcript}"']
        if area:
            if isinstance(area, list):
                params.append(f'area=Region("{area[0]}")')
            else:
                params.append(f'sample_query="country == \'{entities.get("country", "").title()}\'"')
        params.append('min_cohort_size=10')
        
        code = f"ag3.plot_frequencies_heatmap(\n    {(','+chr(10)+'    ').join(params)}\n)"
        explanation = f"Plots allele frequency heatmap for {gene} gene"
        if entities.get("country"):
            explanation += f" in {entities['country'].title()}"
            
    elif method == "plot_frequencies_time_series":
        transcript = entities.get("transcript", "AGAP004707-RD")
        gene = entities.get("gene", "Vgsc")
        area = entities.get("area", None)
        
        params = [f'transcript="{transcript}"']
        if area and not isinstance(area, list):
            params.append(f'area=Region("{area}")')
        params.append('period_by="year"')
        params.append('min_cohort_size=10')
        
        code = f"ag3.plot_frequencies_time_series(\n    {(','+chr(10)+'    ').join(params)}\n)"
        explanation = f"Plots temporal trends of {gene} mutation frequencies"
        if entities.get("country"):
            explanation += f" in {entities['country'].title()}"
            
    elif method == "pairwise_average_fst":
        contig = entities.get("contig", "3L")
        cohort_by = entities.get("cohort_by", "country")
        
        params = [f'contig="{contig}"', f'cohorts="{cohort_by}"', 'min_cohort_size=10']
        code = f"fst_df = ag3.pairwise_average_fst(\n    {(','+chr(10)+'    ').join(params)}\n)"
        explanation = f"Computes pairwise Fst on chromosome {contig} grouped by {cohort_by}"

    elif method == "plot_pairwise_average_fst":
        annotation = entities.get("annotation", None)
        params = ['fst_df']
        if annotation:
            params.append(f'annotation="{annotation}"')
        
        code = f"ag3.plot_pairwise_average_fst(\n    {(','+chr(10)+'    ').join(params)}\n)"
        explanation = f"Plots Fst heatmap"
        if annotation:
            explanation += f" with {annotation} display"
            
    elif method == "sample_metadata":
        params = []
        if entities.get("species"):
            params.append(f'sample_query="taxon == \'{entities["species"]}\'"')
        if entities.get("country"):
            params.append(f'sample_query="country == \'{entities["country"].title()}\'"')
        
        code = f"ag3.sample_metadata(\n    {(','+chr(10)+'    ').join(params) if params else ''}\n)"
        explanation = "Retrieves sample metadata"
        if entities.get("species"):
            explanation += f" for {entities['species']}"
        if entities.get("country"):
            explanation += f" in {entities['country'].title()}"
            
    elif method == "snp_calls":
        contig = entities.get("contig", "2L")
        params = [f'region="{contig}"']
        if entities.get("species"):
            params.append(f'sample_query="taxon == \'{entities["species"]}\'"')
        
        code = f"ag3.snp_calls(\n    {(','+chr(10)+'    ').join(params)}\n)"
        explanation = f"Retrieves SNP genotype data for chromosome {contig}"
        
    else:
        code = "# Could not generate API call — query not understood"
        explanation = "Query could not be mapped to an API method"
    
    return code, explanation

def parse_query(query: str) -> ParsedQuery:
    """Full NLP pipeline: query → intent → entities → API call."""
    intent, confidence = classify_intent(query)
    entities = extract_entities(query)
    api_call, explanation = generate_api_call(intent, entities)
    
    return ParsedQuery(
        raw_query=query,
        intent=intent,
        confidence=confidence,
        entities=entities,
        api_call=api_call,
        explanation=explanation,
    )

print("NLP engine loaded ✅")
print("Components: intent classifier, entity extractor, API call generator")

NLP engine loaded ✅
Components: intent classifier, entity extractor, API call generator


In [5]:
# Cell 5: Demo — Natural Language → API Translation

TEST_QUERIES = [
    "Show me allele frequencies for Vgsc in Kenya",
    "How have kdr mutation frequencies changed over time in Ghana?",
    "Compare genetic differentiation between populations by country on chromosome 3L",
    "Plot the Fst heatmap with lower triangle only",
    "What samples do we have from Tanzania?",
    "Get SNP genotype data for Ace1 in Uganda",
    "Show me the frequency trends for cyp6p3 in Burkina Faso",
    "What is the genetic divergence between gambiae and coluzzii populations?",
    "List all samples of An. arabiensis from East Africa",
    "Visualize insecticide resistance mutations in Mozambique",
]

print("=" * 70)
print("  NATURAL LANGUAGE → malariagen_data API TRANSLATION DEMO")
print("=" * 70)

for i, query in enumerate(TEST_QUERIES, 1):
    result = parse_query(query)
    
    print(f"\n{'─' * 70}")
    print(f"  Query {i}: \"{result.raw_query}\"")
    print(f"{'─' * 70}")
    print(f"  Intent:     {result.intent} (confidence: {result.confidence:.0%})")
    print(f"  Entities:   {result.entities}")
    print(f"  Explanation: {result.explanation}")
    print(f"\n  Generated API call:")
    for line in result.api_call.split('\n'):
        print(f"    {line}")

print(f"\n{'=' * 70}")
print(f"  Processed {len(TEST_QUERIES)} queries successfully")
print(f"{'=' * 70}")

  NATURAL LANGUAGE → malariagen_data API TRANSLATION DEMO

──────────────────────────────────────────────────────────────────────
  Query 1: "Show me allele frequencies for Vgsc in Kenya"
──────────────────────────────────────────────────────────────────────
  Intent:     plot_frequencies_heatmap (confidence: 60%)
  Entities:   {'gene': 'vgsc', 'transcript': 'AGAP004707-RD', 'country': 'kenya', 'area': 'KE'}
  Explanation: Plots allele frequency heatmap for vgsc gene in Kenya

  Generated API call:
    ag3.plot_frequencies_heatmap(
        transcript="AGAP004707-RD",
        sample_query="country == 'Kenya'",
        min_cohort_size=10
    )

──────────────────────────────────────────────────────────────────────
  Query 2: "How have kdr mutation frequencies changed over time in Ghana?"
──────────────────────────────────────────────────────────────────────
  Intent:     plot_frequencies_time_series (confidence: 100%)
  Entities:   {'gene': 'kdr', 'transcript': 'AGAP004707-RD', 'country'

In [6]:
# Cell 6: Results Summary — Accuracy Analysis

results = []
for query in TEST_QUERIES:
    r = parse_query(query)
    results.append(r)

# Print summary table
print(f"{'Query':<55} {'Intent':<35} {'Conf':>5} {'Entities':>3}")
print("─" * 100)
for r in results:
    entity_count = len(r.entities)
    short_query = r.raw_query[:52] + "..." if len(r.raw_query) > 55 else r.raw_query
    print(f"{short_query:<55} {r.intent:<35} {r.confidence:>4.0%} {entity_count:>3}")

# Statistics
avg_confidence = sum(r.confidence for r in results) / len(results)
intents_found = sum(1 for r in results if r.intent != "unknown")
total_entities = sum(len(r.entities) for r in results)

print(f"\n{'─' * 100}")
print(f"Average confidence: {avg_confidence:.0%}")
print(f"Queries resolved:   {intents_found}/{len(results)}")
print(f"Total entities extracted: {total_entities}")

Query                                                   Intent                               Conf Entities
────────────────────────────────────────────────────────────────────────────────────────────────────
Show me allele frequencies for Vgsc in Kenya            plot_frequencies_heatmap             60%   4
How have kdr mutation frequencies changed over time ... plot_frequencies_time_series        100%   4
Compare genetic differentiation between populations ... pairwise_average_fst                 60%   2
Plot the Fst heatmap with lower triangle only           plot_pairwise_average_fst           100%   1
What samples do we have from Tanzania?                  sample_metadata                      20%   2
Get SNP genotype data for Ace1 in Uganda                snp_calls                            80%   4
Show me the frequency trends for cyp6p3 in Burkina Faso plot_frequencies_time_series        100%   4
What is the genetic divergence between gambiae and c... pairwise_average_fst         

In [8]:
# Cell 7: Architecture — From PoC to Production

print("""
══════════════════════════════════════════════════════════════════════════════
  NLP INTERFACE ARCHITECTURE: FROM PoC TO PRODUCTION
══════════════════════════════════════════════════════════════════════════════

  ┌─────────────────────────────────────────────────────────────────────┐
  │                     CURRENT PoC (this notebook)                     │
  │                                                                     │
  │   User Query ──→ Keyword Intent ──→ Regex Entity ──→ Template Code  │
  │    (string)      Classifier         Extractor        Generator      │
  │                                                                     │
  │   Strengths: Fast, interpretable, zero dependencies                 │
  │   Limits:    Rigid patterns, no paraphrasing, no disambiguation     │
  └─────────────────────────────────────────────────────────────────────┘
                                  │
                                  ▼
  ┌─────────────────────────────────────────────────────────────────────┐
  │                  PROPOSED GSoC IMPLEMENTATION                       │
  │                                                                     │
  │   ┌──────────┐    ┌──────────────┐    ┌────────────┐    ┌────────┐  │
  │   │  Query   │    │  LLM-based   │    │  API Call  │    │Response│  │
  │   │  Parser  │───→│  Intent +    │───→│  Generator │───→│Format- │  │
  │   │(tokenize)│    │  Entity NER  │    │(validated) │    │  ter   │  │
  │   └──────────┘    └──────────────┘    └────────────┘    └────────┘  │
  │        ▼                ▼                    ▼               ▼      │
  │   Spell-check     Fine-tuned LLM      Type-check       Plotly/      │
  │   + normalize     (LoRA on domain     against API      pandas       │
  │                    examples) OR        signatures       summary     │
  │                    RAG + few-shot      from docstrings              │
  │                    prompting                                        │
  │                                                                     │
  │   Key advances over PoC:                                            │
  │   • Handles paraphrasing ("insecticide resistance" → Vgsc/Rdl)      │
  │   • Disambiguates multi-step queries                                │
  │   • Validates parameters against actual API type annotations        │
  │   • Returns error explanations for invalid queries                  │
  │   • Learns from user feedback                                       │
  └─────────────────────────────────────────────────────────────────────┘

  LEVERAGING EXISTING INFRASTRUCTURE:
  ──────────────────────────────────
  • describe_api() (PR #904)  → Method-level discovery for intent mapping
  • @_check_types decorator   → Parameter validation at runtime
  • Annotated type aliases    → Extractable parameter descriptions
  • Docstrings                → Grounding data for LLM/RAG context

  RELEVANT PRIOR WORK:
  ──────────────────────────────────
  • llama-task-agent           → LoRA fine-tuned LLaMA for NL→structured
                                 task execution (100% format compliance)
  • biodiversity-publication-  → SciBERT fine-tuned on genomics literature
    analyzer                     (236 domain terms, 99.5% F1)
  • complaint-intelligence-    → RAG pipeline with FAISS + embeddings
    system                       over 12M+ records

══════════════════════════════════════════════════════════════════════════════
""")


══════════════════════════════════════════════════════════════════════════════
  NLP INTERFACE ARCHITECTURE: FROM PoC TO PRODUCTION
══════════════════════════════════════════════════════════════════════════════

  ┌─────────────────────────────────────────────────────────────────────┐
  │                     CURRENT PoC (this notebook)                     │
  │                                                                     │
  │   User Query ──→ Keyword Intent ──→ Regex Entity ──→ Template Code  │
  │    (string)      Classifier         Extractor        Generator      │
  │                                                                     │
  │   Strengths: Fast, interpretable, zero dependencies                 │
  │   Limits:    Rigid patterns, no paraphrasing, no disambiguation     │
  └─────────────────────────────────────────────────────────────────────┘
                                  │
                                  ▼
  ┌──────────────────────────────────────────────

In [9]:
# Cell 8: Interactive Query Interface

def interactive_demo():
    """Run an interactive query session."""
    print("\n🦟 MalariaGEN NLP Interface — Interactive Demo")
    print("   Type a question about mosquito genomic data.")
    print("   Type 'quit' to exit.\n")
    
    example_queries = [
        "Show me allele frequencies for Vgsc in Kenya",
        "How have kdr mutations changed over time in Ghana?",
        "Compare Fst between populations on chromosome 2L by country",
        "What samples do we have from Burkina Faso?",
    ]
    print("   Example queries:")
    for eq in example_queries:
        print(f"     • {eq}")
    print()
    
    while True:
        try:
            query = input("   🔍 Your question: ").strip()
        except (EOFError, KeyboardInterrupt):
            break
            
        if query.lower() in ('quit', 'exit', 'q'):
            break
        if not query:
            continue
            
        result = parse_query(query)
        
        print(f"\n   📋 Intent:      {result.intent}")
        print(f"   🎯 Confidence:  {result.confidence:.0%}")
        print(f"   🔬 Entities:    {result.entities}")
        print(f"   💬 Explanation: {result.explanation}")
        print(f"\n   🐍 Generated code:")
        for line in result.api_call.split('\n'):
            print(f"      {line}")
        print()

# Uncomment to run interactively:
# interactive_demo()

# Non-interactive demonstration
print("Interactive demo available — uncomment interactive_demo() to try.\n")
print("Example session:")
print('   🔍 Your question: "Show me resistance mutations in Cameroon"')
result = parse_query("Show me resistance mutations in Cameroon")
print(f"   📋 Intent:      {result.intent}")
print(f"   🎯 Confidence:  {result.confidence:.0%}")
print(f"   🔬 Entities:    {result.entities}")
print(f"   💬 Explanation: {result.explanation}")
print(f"\n   🐍 Generated code:")
for line in result.api_call.split('\n'):
    print(f"      {line}")

Interactive demo available — uncomment interactive_demo() to try.

Example session:
   🔍 Your question: "Show me resistance mutations in Cameroon"
   📋 Intent:      plot_frequencies_heatmap
   🎯 Confidence:  40%
   🔬 Entities:    {'country': 'cameroon', 'area': 'CM'}
   💬 Explanation: Plots allele frequency heatmap for Vgsc gene in Cameroon

   🐍 Generated code:
      ag3.plot_frequencies_heatmap(
          transcript="AGAP004707-RD",
          sample_query="country == 'Cameroon'",
          min_cohort_size=10
      )


In [10]:
# Cell 9: Summary & Conclusion

print("""
══════════════════════════════════════════════════════════════════════════════
  NLP INTERFACE PoC — RESULTS SUMMARY
══════════════════════════════════════════════════════════════════════════════

  PERFORMANCE (10 test queries)
  ─────────────────────────────
  Queries resolved:        10/10 (100%)
  Average confidence:      62%
  Total entities extracted: 29

  CAPABILITIES DEMONSTRATED
  ─────────────────────────
  ✅ Gene name → transcript ID mapping    (vgsc → AGAP004707-RD)
  ✅ Country → area code resolution       (kenya → KE)
  ✅ Temporal query detection             ("over time" → time_series)
  ✅ Population comparison parsing        ("compare...by country" → Fst)
  ✅ Species name normalization           (An. arabiensis → arabiensis)
  ✅ Annotation preference extraction     ("lower triangle" → annotation)
  ✅ Multi-entity extraction              (gene + country in one query)
  ✅ Executable Python code generation    (valid malariagen_data calls)

  KNOWN LIMITATIONS (PoC scope)
  ─────────────────────────────
  ⚠ Keyword-based intent — no paraphrase handling
  ⚠ No disambiguation for ambiguous queries
  ⚠ No multi-step query chaining (e.g., "compute Fst then plot it")
  ⚠ No parameter validation against live API signatures
  ⚠ Fixed entity dictionaries — no learning from new data

  PROPOSED GSoC ENHANCEMENTS
  ──────────────────────────
  1. Replace keyword classifier with fine-tuned LLM (LoRA on domain
     examples, leveraging llama-task-agent architecture)
  2. Use RAG over API docstrings + training course notebooks for
     grounding (leveraging complaint-intelligence-system RAG pipeline)
  3. Validate generated calls against @_check_types decorator and
     Annotated type aliases at runtime
  4. Build evaluation suite: 50+ queries with expected API calls
  5. Add conversational context for multi-turn analysis sessions

  RELEVANT PRIOR WORK BY AUTHOR
  ─────────────────────────────
  • llama-task-agent    — LLaMA-3.1-8B fine-tuned with LoRA for
                          NL → structured task execution (100% format
                          compliance). Architecture directly applicable.
  • biodiversity-       — SciBERT on genomics literature (236 domain
    publication-analyzer  terms, 99.5% F1). Domain vocabulary transferable.
  • complaint-          — RAG pipeline with FAISS + Gemini over 12M+
    intelligence-system   records. Retrieval architecture applicable.

  Author: Aswani Sahoo (@AswaniSahoo)
══════════════════════════════════════════════════════════════════════════════
""")


══════════════════════════════════════════════════════════════════════════════
  NLP INTERFACE PoC — RESULTS SUMMARY
══════════════════════════════════════════════════════════════════════════════

  PERFORMANCE (10 test queries)
  ─────────────────────────────
  Queries resolved:        10/10 (100%)
  Average confidence:      62%
  Total entities extracted: 29

  CAPABILITIES DEMONSTRATED
  ─────────────────────────
  ✅ Gene name → transcript ID mapping    (vgsc → AGAP004707-RD)
  ✅ Country → area code resolution       (kenya → KE)
  ✅ Temporal query detection             ("over time" → time_series)
  ✅ Population comparison parsing        ("compare...by country" → Fst)
  ✅ Species name normalization           (An. arabiensis → arabiensis)
  ✅ Annotation preference extraction     ("lower triangle" → annotation)
  ✅ Multi-entity extraction              (gene + country in one query)
  ✅ Executable Python code generation    (valid malariagen_data calls)

  KNOWN LIMITATIONS (PoC scope)
 